# Data overview: homes, visits, population and income

Starting point for the third-places project. It shows the data you can build on; the analysis is yours.

**Twitter tables** (PostgreSQL `twitter_cities_v2`, schema `revised`; built by `code/05_home_processing/home_pipeline/sql/`,
every rule documented in `home_pipeline/DECISIONS.md`):

| table | one row per | what it is |
|---|---|---|
| `home_final` | (city, user_id, res) | home H3 cell of a user, res 10 (~65 m) and 9 (~175 m); `confidence` high / medium |
| `work_final` | (city, user_id) | work cell (res 10), only where a clear weekday 09-17 place exists; most users have none |
| `visit` | (city, user_id, local_date, cell_10) | **third-place candidates**: places a home user tweeted from on a day, home area and work cell removed |
| `visit_stay` | (city, user_id, local_date, cell_10) | the same but with home and work stays still in |

All tweets are 2013-2017, GPS-tagged, inside the city box, no retweets; times are local. Only users with a home
are in `visit`. A visit = a user was in a res 10 cell on a day (several tweets that day count once, `n_tweets`).
`max_point_users` is set when the exact coordinate is shared by many users: usually a check-in venue (Swarm /
Foursquare), sometimes an airport or station. `dist_home_m` = distance from the user's home cell.

**External data** (repo root `external_data/`, each folder has a README with sources and method):

| file | content |
|---|---|
| `population/pop_h3.csv` | residents per H3 cell, res 8/9/10, all three cities (GHS-POP 2015) |
| `income/income_h3.csv` | household income per H3 cell, res 8/9/10, **London only** (ONS MSOA estimates 2015/16) |
| `income/msoa_income_2016.csv` | the same income per MSOA (the level it is measured at) |

Credentials: `code/03_database_setup/connection.json`.

In [ ]:
# ── imports ───────────────────────────────────────────────────────────────────
import json
from pathlib import Path

import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from shapely.geometry import Polygon
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

FS = 10
FS2 = FS - 2
FS4 = FS - 4
plt.rcParams.update({"font.size": FS})

# ── paths ─────────────────────────────────────────────────────────────────────
EXT = Path("../../external_data")
POP_CSV = EXT / "population" / "pop_h3.csv"
INCOME_CSV = EXT / "income" / "income_h3.csv"
MSOA_CSV = EXT / "income" / "msoa_income_2016.csv"

# ── connection ────────────────────────────────────────────────────────────────
cfg = json.load(open(Path("../03_database_setup/connection.json")))
url = URL.create(
    "postgresql+psycopg2",
    username=cfg["user"],
    password=cfg["password"],
    host=cfg["host"],
    port=cfg["port"],
    database=cfg["database"],
)
# public on the path for the h3index operators and functions
engine = create_engine(url, connect_args={"options": "-c search_path=revised,public"})

# ── cities ────────────────────────────────────────────────────────────────────
CITIES = ["amsterdam", "london", "portland"]
CITY_LABEL = {"amsterdam": "Amsterdam", "london": "London", "portland": "Portland"}

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
def hex_gdf(df):
    """DataFrame with an H3 `cell` column -> GeoDataFrame of hexagons in EPSG:3857."""
    geom = [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in df["cell"]]
    return gpd.GeoDataFrame(df, geometry=geom, crs="EPSG:4326").to_crs(epsg=3857)


def hex_map(ax, gdf, column, norm, cbar_label, label, cmap="Blues"):
    """Hexagons coloured by `column` on a light basemap, colorbar matched to the axis."""
    gdf.plot(column=column, cmap=cmap, norm=norm, linewidth=0, alpha=0.85, ax=ax)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=FS4)
    ax.set_axis_off()
    ax.text(0.0, 1.02, label, transform=ax.transAxes, fontsize=FS, fontweight="bold")
    cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.05)
    cb = plt.gcf().colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=cax)
    cb.set_label(cbar_label, fontsize=FS2)
    cb.ax.tick_params(labelsize=FS2)

---
# 1. Population

Residents per H3 cell (GHS-POP 2015, 100 m grid aggregated to H3). Covers the whole city box, which is larger
than the city itself (London box 9.8M vs Greater London 8.6M). Res 10 values are noisy (a cell holds ~1.5 grid
pixels); res 9 or 8 is safer for anything per cell.

In [ ]:
# ── load ──────────────────────────────────────────────────────────────────────
pop = pd.read_csv(POP_CSV, dtype={"cell": str})
pop.groupby(["city", "res"])["pop"].agg(["count", "sum", "median"]).round(0)

In [ ]:
# ── map: residents per res 9 cell, one panel per city ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, city, panel in zip(axes, CITIES, ["(A)", "(B)", "(C)"]):
    g = hex_gdf(pop[(pop["city"] == city) & (pop["res"] == 9)])
    hex_map(ax, g, "pop", LogNorm(1, g["pop"].max()), "residents per H3 res 9 cell",
            f"{panel} {CITY_LABEL[city]}", cmap="Greens")
plt.tight_layout()

---
# 2. Income (London)

ONS model-based estimates of mean household income per MSOA (~8,000 residents each), financial year 2015/16.
Four measures: `income_total`, `income_net`, and net equivalised income before / after housing costs
(`income_net_bhc`, `income_net_ahc`; equivalised = adjusted for household size and composition, usually the
preferred measure for comparing living standards). The H3 values are population-weighted blends of the MSOAs a cell
overlaps, so the real resolution is the MSOA: finer cells do not add income detail. Estimates have wide intervals
(see the `_ci` columns in `msoa_income_2016.csv`), so income groups (quartiles, deciles) are safer than exact values.
`greater_london` marks cells whose population lies mostly in Greater London.

In [ ]:
# ── load ──────────────────────────────────────────────────────────────────────
msoa = pd.read_csv(MSOA_CSV)
inc = pd.read_csv(INCOME_CSV, dtype={"cell": str})
n_gl = (msoa["region_cd"] == "E12000007").sum()
print(f"MSOAs in the London box: {len(msoa)} ({n_gl} in Greater London)")
msoa[["income_total", "income_net", "income_net_bhc", "income_net_ahc"]].describe().round(0)

In [ ]:
# ── map: net equivalised income before housing costs, res 9 ───────────────────
g = hex_gdf(inc[inc["res"] == 9])
fig, ax = plt.subplots(figsize=(9, 6.5))
hex_map(ax, g, "income_net_bhc", Normalize(g["income_net_bhc"].quantile(0.02), g["income_net_bhc"].quantile(0.98)),
        "net equivalised household income before housing costs (GBP, 2015/16)", "(A) London, res 9",
        cmap="viridis")
plt.tight_layout()

In [ ]:
# ── distribution across MSOAs: Greater London vs the rest of the box ──────────
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.arange(20000, 70001, 2000)
gl = msoa["region_cd"] == "E12000007"
ax.hist(msoa.loc[gl, "income_net_bhc"], bins=bins, color="#2a78d6", alpha=0.7, label="Greater London")
ax.hist(msoa.loc[~gl, "income_net_bhc"], bins=bins, color="#eb6834", alpha=0.7, label="rest of the box")
ax.set_xlabel("net equivalised household income before housing costs (GBP)", fontsize=FS2)
ax.set_ylabel("MSOAs", fontsize=FS2)
ax.tick_params(labelsize=FS2)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=FS2)
plt.tight_layout()

## 2.1 Income groups

One way to define income groups: population-weighted quartiles of res 9 cells inside Greater London, so each
quartile holds a quarter of the residents. The cut points below are an example; the choice (measure, deciles vs
quartiles, Greater London only or the whole box) is part of your design.

In [ ]:
# ── population-weighted quartile cut points, Greater London, res 9 ────────────
gl9 = inc[(inc["res"] == 9) & inc["greater_london"]].sort_values("income_net_bhc")
cum = gl9["pop"].cumsum() / gl9["pop"].sum()
cuts = [gl9.loc[cum >= q, "income_net_bhc"].iloc[0] for q in [0.25, 0.5, 0.75]]
gl9["income_q"] = 1 + np.searchsorted(cuts, gl9["income_net_bhc"], side="right")
print("cut points (GBP):", [int(c) for c in cuts])
gl9.groupby("income_q").agg(cells=("cell", "count"), pop=("pop", "sum"),
                            income_min=("income_net_bhc", "min"),
                            income_max=("income_net_bhc", "max")).round(0)

---
# 3. Twitter tables

Size of the home, work and visit tables per city. The SQL cells aggregate in Postgres; never pull whole tables
(`visit` has millions of rows).

In [ ]:
# ── homes, work places and visits per city ────────────────────────────────────
pd.read_sql("""
select h.city
     , count(*)                                                as n_home_users
     , count(*) filter (where h.confidence = 'high')           as n_high
     , (select count(*) from revised.work_final w
         where w.city = h.city)                                as n_work
     , (select count(*) from revised.visit v
         where v.city = h.city)                                as n_visits
     , (select count(distinct cell_10) from revised.visit v
         where v.city = h.city)                                as n_visited_cells
from revised.home_final h
where h.res = 9
group by h.city
order by h.city
""", engine)

## 3.1 Example join: home income of London users

Homes join to income on the H3 cell (`home_final.cell` at res 9 = `income_h3.cell` at res 9). Here: how home users
spread over the income quartiles of section 2.1, next to the residents. Twitter users are not a random sample
(~1-2% of residents, younger, heavier users in the centre), so expect a skew.

In [ ]:
# ── London homes at res 9 ─────────────────────────────────────────────────────
homes = pd.read_sql("""
select user_id, cell::text as cell, confidence
from revised.home_final
where city = 'london' and res = 9
""", engine)

# ── join to income groups (Greater London cells only) ─────────────────────────
homes = homes.merge(gl9[["cell", "income_net_bhc", "income_q"]], on="cell", how="inner")
pd.DataFrame({
    "share_homes": homes["income_q"].value_counts(normalize=True).sort_index(),
    "share_residents": gl9.groupby("income_q")["pop"].sum() / gl9["pop"].sum(),
}).round(3)

## 3.2 Example: visitors per place

A visit row carries the visitor's `user_id`; the visitor's home (and so income) comes from `home_final`. Example:
the res 9 cells with the most distinct visitors in London. Matching cells to amenities (POIs) is the next step.

In [ ]:
# ── most visited res 9 cells in London ────────────────────────────────────────
pd.read_sql("""
select public.h3_cell_to_parent(cell_10, 9)::text    as cell_9
     , count(distinct user_id)                       as n_visitors
     , count(*)                                      as n_visits
     , round(avg((max_point_users is not null)::int)::numeric, 2) as share_at_shared_point
from revised.visit
where city = 'london'
group by 1
order by n_visitors desc
limit 10
""", engine)

---
# 4. Caveats to keep in mind

- **Sample:** homes exist for ~150k users (~1-2% of residents); heavy users and central areas are over-represented.
- **Activity:** the number of places a user visits grows with how much they tweet. Any per-user count (places,
  distance travelled, diversity) must be normalised by active days, or computed on a fixed random sample of days
  per user; otherwise income effects can just be tweeting effects. `08_visits.sql` V2 shows the size of this effect.
- **Visit = a tweet location**, not a confirmed stay at a venue. GPS error is 10-50 m, so a res 10 cell (~65 m)
  is the finest sensible unit; match POIs to cells (or small buffers), not tweets to single venues.
- **Home area removed:** the home cell and its neighbours (~200 m) are not in `visit`; the local corner shop is
  therefore invisible.
- **Income** is measured per MSOA and is a model estimate; use groups, not exact values.